# Compress a Spark DataFrame when loading it into pandas

[All How-to Guides](README.md)

If the next step needs the pandas backend, pass the Spark SQL DataFrame directly to `Dataset` with `backend=BackendsEnum.pandas`. `data_compression` controls the transformations applied on Spark before the driver collects the result.

For local Spark, set `JAVA_HOME` to a Java 17 installation. Test the conversion on a small, representative sample first. For a large input, select the rows and columns you need in Spark before collecting them.

## Prepare a Spark DataFrame

Create a local table with 50,000 rows. Use a double-precision `metric` and a string `segment` with repeated categories. Two shuffle partitions are enough for this sample.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from hypex.dataset import Dataset, TargetRole
from hypex.utils import BackendsEnum

spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx Spark to pandas")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark_frame = spark.range(50_000).select(
    F.col("id").alias("event_id"),
    (F.col("id") / 10).cast("double").alias("metric"),
    F.when(F.col("id") % 2 == 0, "new_customer")
    .otherwise("returning_customer")
    .alias("segment"),
)
spark_frame.dtypes

[('event_id', 'bigint'), ('metric', 'double'), ('segment', 'string')]

The schema contains `bigint`, `double`, and `string`. Watch how compression affects the types of the metric and segment columns.

## Load the data with compression

Pass the Spark DataFrame to the `Dataset` constructor. The example spells out the default `data_compression="auto"` to show where to change it. Display the selected backend and the labels available through `labels_dict`.

In [2]:
compact = Dataset(
    roles={"metric": TargetRole()},
    data=spark_frame,
    backend=BackendsEnum.pandas,
    data_compression="auto",
)
print("Backend:", compact.backend_type.value)
compact.labels_dict

Backend: pandas


{'segment': {0: 'new_customer', 1: 'returning_customer'}}

`Backend: pandas` confirms the backend. `segment` stores integer codes, and `labels_dict` maps each code to its original label. To exclude a column from encoding, give it an explicit string role type, such as `"segment": InfoRole(str)`. Modes with encoding fill missing string values with `UNKNOWN`.

## Compare modes and memory use

- `"disable"` collects the table into pandas as-is. Use it as a baseline on a small sample.
- `"downcasting"` converts `double` and `decimal` columns to `float32`. Check that the reduced precision is suitable for your metric.
- `"encoding"` replaces eligible strings with integer codes and preserves numeric precision.
- `"auto"` applies both transformations.

Compare all four modes on the sample. Access the pandas DataFrame through the public `Dataset.data` property and measure its memory in MiB, including the strings.

In [3]:
datasets = {
    mode: Dataset(
        roles={"metric": TargetRole()},
        data=spark_frame,
        backend=BackendsEnum.pandas,
        data_compression=mode,
    )
    for mode in ("disable", "downcasting", "encoding")
}
datasets["auto"] = compact

measurements = {}
for mode, dataset in datasets.items():
    table = dataset.data
    measurements[mode] = {
        "memory (MiB)": table.memory_usage(deep=True).sum() / 1024**2,
        "metric dtype": str(table["metric"].dtype),
        "segment dtype": str(table["segment"].dtype),
    }

pd.DataFrame.from_dict(measurements, orient="index").round(2)

,memory (MiB),metric dtype,segment dtype
disable,4.20,float64,object
downcasting,4.01,float32,object
encoding,0.95,float64,int32
auto,0.76,float32,int32


`auto` reduced the final DataFrame from 4.20 to 0.76 MiB. Encoding the repeated strings accounted for most of the savings on this sample. Savings depend on data types, distributions, and the number of distinct categories.

`memory_usage(deep=True)` measures the final DataFrame. Budget extra driver memory for collection and conversion. If the result still exceeds local memory, keep the computation on Spark. Set `data_compression` on the `Dataset` constructor when passing the source Spark DataFrame. `to_backend()` on an existing Spark-backed `Dataset` has its own conversion path.

Stop the local session created for this example.

In [4]:
spark.stop()
print("Spark session stopped")

Spark session stopped
